In [0]:
# COMMAND ---------- NOTEBOOK

# CREATE VOLUME IF NOT EXISTS dev.stepright.staging;


dbutils.widgets.text("batch", "0", "Batch number to load")
batch_num = dbutils.widgets.get("batch")
 
staging_root = f"/Volumes/dev/stepright/staging/batch_{batch_num}"
landing_root = "/Volumes/dev/stepright/landing"
 
print(f"Loading batch_{batch_num} from {staging_root} into {landing_root}")


subfolders = [
    "orders_cdc",
    "order_items_cdc",
    "customers_cdc",
    "products",
    "categories",
    "clickstream",
    "inventory",
]
 
def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False
 
 
for folder in subfolders:
    src = f"{staging_root}/{folder}"
    dst = f"{landing_root}/{folder}"
    if path_exists(src):
        dbutils.fs.cp(src, dst, recurse=True)
        print(f"Copied: {src} -> {dst}")
    else:
        print(f"Skipped {folder} — nothing staged for batch_{batch_num}. "
              f"Expected for products/categories on most incremental batches — "
              f"see data_generator.py for which sources each batch touches.")


csv_sources = ["products", "categories", "inventory"]
json_sources = ["orders_cdc", "order_items_cdc", "customers_cdc", "clickstream"]
 
print(f"{'source':<20}{'format':<8}{'row_count'}")
print("-" * 45)
 
for folder in csv_sources:
    path = f"{landing_root}/{folder}"
    try:
        count = spark.read.option("header", "true").csv(path).count()
    except Exception:
        count = 0
    print(f"{folder:<20}{'csv':<8}{count}")
 
for folder in json_sources:
    path = f"{landing_root}/{folder}"
    try:
        count = spark.read.json(path).count()
    except Exception:
        count = 0
    print(f"{folder:<20}{'json':<8}{count}")
 

